# 🔥 Google Colab - Crypto Trader Training Pipeline

Este notebook ejecuta el pipeline completo de entrenamiento:
1. Configuración del entorno
2. Descarga de datos históricos
3. Feature engineering
4. Entrenamiento del modelo
5. Evaluación y backtest

**Nota**: Los datos y modelos se guardan en el almacenamiento temporal de Colab. Para persistencia, descarga los archivos al final.

In [ ]:
# @title 📦 Paso 1: Instalar dependencias
# @markdown Ejecuta esta celda para instalar todas las librerías necesarias.
!pip install -q ccxt pandas numpy lightgbm scikit-learn joblib loguru vectorbt matplotlib pandas-ta

In [ ]:
# @title 🛠️ Paso 2: Configurar entorno
# @markdown Clona el repositorio (borra el anterior si existe) y configura las variables de entorno.
import os
import subprocess
import shutil

# Cambiar a un directorio seguro antes de borrar
os.chdir("/content")

# Borrar directorio existente si existe
if os.path.exists("/content/trader-ia-v2"):
    shutil.rmtree("/content/trader-ia-v2")
    print("🗑️ Directorio anterior eliminado")

# Clonar repositorio
!git clone https://github.com/hoserpa/trader-ia-v2.git /content/trader-ia-v2
os.chdir("/content/trader-ia-v2")

# Configurar PYTHONPATH para imports locales
os.environ["PYTHONPATH"] = "/content/trader-ia-v2"

print("✅ Entorno configurado")

In [ ]:
# @title 📥 Paso 3: Descargar datos históricos
# @markdown Descarga datos de Coinbase Advanced Trade.
# @param {type:"string"} pairs="BTC-EUR,ETH-EUR"
# @param {type:"string"} timeframe="5m"
# @param {type:"number"} days=90
pairs = "BTC-EUR,ETH-EUR"  # @param {type:"string"}
timeframe = "5m"  # @param {type:"string"}
days = 90  # @param {type:"number"}

!python training/fetch_historical_data.py \
  --pairs {pairs} \
  --timeframe {timeframe} \
  --days {days} \
  --output output/data

In [ ]:
# @title 🔧 Paso 4: Feature Engineering
# @markdown Calcula indicadores técnicos y genera features.
# @param {type:"number"} lookahead=3
# @param {type:"number"} threshold=0.015
lookahead = 3  # @param {type:"number"}
threshold = 0.015  # @param {type:"number"}

!python training/feature_engineering.py \
  --data output/data \
  --output output/features \
  --lookahead {lookahead} \
  --threshold {threshold}

In [ ]:
# @title 🤖 Paso 5: Entrenar Modelo
# @markdown Entrena el modelo LightGBM con los features generados.
!python training/train_model.py \
  --data output/features/features_with_labels.parquet \
  --output output/model

In [ ]:
# @title 📊 Paso 6: Evaluar Modelo
# @markdown Ejecuta evaluación completa con backtest.
!python training/evaluate_model.py \
  --model output/model/trained_model.pkl \
  --scaler output/model/scaler.pkl \
  --metadata output/model/model_metadata.json \
  --data output/features/features_with_labels.parquet \
  --output output/evaluation \
  --initial-cash 10000

In [ ]:
# @title ⬇️ Descargar archivos
# @markdown Ejecuta esta celda para descargar los modelos y resultados.
from google.colab import files
import os

def download_folder(folder_path, zip_name):
    if os.path.exists(folder_path):
        !zip -r {zip_name} {folder_path}
        files.download(zip_name)
    else:
        print(f"⚠️ Carpeta no encontrada: {folder_path}")

# Descargar modelo
print("📦 Descargando modelo entrenado...")
download_folder("output/model", "model.zip")

# Descargar evaluación
print("📊 Descargando resultados de evaluación...")
download_folder("output/evaluation", "evaluation.zip")

In [ ]:
# @title 📈 Ver resultados
# @markdown Muestra las métricas de evaluación.
import json

with open("output/evaluation/evaluation_results.json", "r") as f:
    results = json.load(f)

print("=" * 50)
print("📊 MÉTRICAS DE CLASIFICACIÓN")
print("=" * 50)
for k, v in results["classification_metrics"].items():
    print(f"  {k}: {v:.4f}")

print("\n" + "=" * 50)
print("📈 ESTADÍSTICAS DEL BACKTEST")
print("=" * 50)
for k, v in results["backtest_stats"].items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# @title 🖼️ Ver gráficos
# @markdown Muestra la matriz de confusión y curvas ROC.
from IPython.display import Image, display

print("📊 Matriz de Confusión:")
display(Image("output/evaluation/confusion_matrix.png"))

print("\n📈 Curvas ROC:")
display(Image("output/evaluation/roc_curves.png"))